# Text orientation — Colab training

Короткий воспроизводимый запуск pretrained MobileNetV3, EfficientNet-B0 или прямоугольного ViT-B/16. Для первого запуска оставьте `QUICK_RUN = True`. ZIP сохраняется в `training/runs`, recovery — в `training/recovery`, а полный запуск может обновить самодостаточный champion bundle в `registry`.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
MODEL = "mobilenet"  # mobilenet | efficientnet | vit
QUICK_RUN = True
PROMOTE_CHAMPION = not QUICK_RUN
RUN_CALIBRATION = not QUICK_RUN
RESUME_TRAINING = True
RUN_TESTS = True
TRAIN_BATCH_SIZE = None  # None = architecture default; ViT OOM fallback: 8
VALIDATION_BATCH_SIZE = None  # None = architecture default; ViT OOM fallback: 16
PROJECT_DIR = "/content/drive/MyDrive/text-orientation"

In [ ]:
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/content/text-orientation-classification")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("В Colab выберите Runtime → Change runtime type → T4 GPU")
print("gpu:", torch.cuda.get_device_name(0))
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
model_settings = {
    "mobilenet": ("configs/baseline.yaml", "mobilenet_v3_large", 64, 128),
    "efficientnet": ("configs/efficientnet_b0.yaml", "efficientnet_b0", 64, 128),
    "vit": ("configs/vit_b_16.yaml", "vit_b_16", 16, 32),
}
if MODEL not in model_settings:
    raise ValueError("MODEL must be 'mobilenet', 'efficientnet', or 'vit'")
config, model_registry_name, quick_batch_size, quick_validation_batch_size = model_settings[MODEL]
run_name = f"colab_{MODEL}_{'quick' if QUICK_RUN else 'full'}"
command = [sys.executable, "-m", "scripts.train", "--config", config, "--run-name", run_name]
run_mode = "quick" if QUICK_RUN else "full"
project_dir = Path(PROJECT_DIR)
registry_dir = project_dir / "registry"
recovery_dir = project_dir / "training" / "recovery" / model_registry_name / run_mode
command += ["--recovery-dir", str(recovery_dir)]
if RESUME_TRAINING:
    command.append("--resume")
if QUICK_RUN:
    train_batch_size = TRAIN_BATCH_SIZE or quick_batch_size
    validation_batch_size = VALIDATION_BATCH_SIZE or quick_validation_batch_size
    command += [
        "--train-base-samples", "2048",
        "--validation-base-samples", "512",
        "--frozen-epochs", "1",
        "--finetune-epochs", "2",
        "--batch-size", str(train_batch_size),
        "--validation-batch-size", str(validation_batch_size),
    ]
else:
    if TRAIN_BATCH_SIZE is not None:
        command += ["--batch-size", str(TRAIN_BATCH_SIZE)]
    if VALIDATION_BATCH_SIZE is not None:
        command += ["--validation-batch-size", str(VALIDATION_BATCH_SIZE)]

run_dir = Path("artifacts/experiments") / run_name
run_dir.mkdir(parents=True, exist_ok=True)
log_path = run_dir / "colab.log"
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code != 0:
    hint = " For ViT CUDA OOM, set TRAIN_BATCH_SIZE=8 and VALIDATION_BATCH_SIZE=16." if MODEL == "vit" else ""
    raise RuntimeError(f"Training failed with exit code {return_code}.{hint}")
if RUN_CALIBRATION:
    subprocess.run([
        sys.executable, "-m", "scripts.calibrate",
        "--run-dir", str(run_dir),
        "--config", config,
    ], check=True)

In [ ]:
import json, platform, shutil
from datetime import datetime, timezone

environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
}
(run_dir / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
project_dir.mkdir(parents=True, exist_ok=True)
runs_output = project_dir / "training" / "runs" / model_registry_name / run_mode
runs_output.mkdir(parents=True, exist_ok=True)
archive_base = Path("/content") / run_name
archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=run_dir))
destination = runs_output / archive.name
shutil.copy2(archive, destination)
if PROMOTE_CHAMPION:
    subprocess.run([
        sys.executable, "-m", "scripts.promote_champion",
        "--run-dir", str(run_dir),
        "--registry-dir", str(registry_dir),
    ], check=True)
else:
    print("Quick run archived but not promoted to champion.")
print("Готово. Пришлите этот файл:", destination)
print("Содержимое:", sorted(path.name for path in run_dir.iterdir()))